Creating our own instruction dataset
------------------------------------

To create a high-quality instruction dataset, we need to address two main issues:

1. The unstructured nature of our data
2. The limited number of articles we can crawl

# Setup and Imports

In [ ]:
import concurrent.futures
import json
import random
import re
from concurrent.futures import ThreadPoolExecutor

from typing import List, Tuple
from datasets import Dataset
from openai import OpenAI
from pydantic import BaseModel, Field
from tqdm.auto import tqdm

import os
from dotenv import load_dotenv
from huggingface_hub import login

In [ ]:
# constants

MODEL = 'gpt-4o-mini'
DATASET = '../content/stoops.twin.articles.json'

In [ ]:
# set up environment

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

os.environ['HF_TOKEN'] = os.getenv('HF_TOKEN')

In [ ]:
# Log in to HuggingFace

hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

# Create Dataset

The raw data we have is a JSON file. We create a Hugging Face dataset from this JSON file by extracting specific fields from each article: id, content, platform, author_id, author name, and link.

Note: this is setup to read from a JSON file exported from MongoDB from the Compass application.

In [ ]:
def load_articles_from_json(file_path: str) -> Dataset:
	with open(file_path, "r", encoding='utf-8', errors='ignore') as file:
		data = json.load(file)
		
	return Dataset.from_dict(
		{
			"id": [item["_id"] for item in data],
			"content": [item["content"]["Content"] for item in data],
			"platform": [item["platform"] for item in data],
			"author_id": [item["author_id"] for item in data],
			"author_full_name": [item["author_full_name"] for item in data],
			"link": [item["link"] for item in data],
		}
	)

In [ ]:
with open(DATASET, "r", encoding='utf-8', errors='ignore') as file:
    data = json.load(file)

print(data)

# Clean Data

Use regex to:

- remove non-alphanumeric characters except for apostrophes, periods, commas, exclamation marks, and question marks
- replace multiple consecutive whitespace characters with a single space

And implement strip() to remove any leading or trailing whitespace.

In [ ]:
def clean_text(text):
    text = re.sub(r"[^\w\s.,!?']", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# Chunk Articles

The `extract_substrings` function processes each article in the dataset by first cleaning the text and then using a regex to split it into sentences. It then builds chunks of text by concatenating these sentences until each chunk is between 1,000 and 2,000 characters long (optimize depending on the density of the information contained in the text by overriding defaults).

In [ ]:
def extract_substrings(dataset: Dataset, min_length: int = 1000, max_length: int = 2000) -> List[str]:
	extracts = []
	sentence_pattern = r"(?<!\w\.\w.)(?<![A-Z][a-z]\.)(?<=\.|\?|\!)\s"
	
	for article in dataset["content"]:
		cleaned_article = clean_text(article)
		sentences = re.split(sentence_pattern, cleaned_article)
		
		current_chunk = ""
		for sentence in sentences:
			sentence = sentence.strip()
			if not sentence:
				continue
				
			if len(current_chunk) + len(sentence) <= max_length:
				current_chunk += sentence + " "
			else:
				if len(current_chunk) >= min_length:
					extracts.append(current_chunk.strip())
				current_chunk = sentence + " "
		
		if len(current_chunk) >= min_length:
			extracts.append(current_chunk.strip())
			
	return extracts

# Create Instruction-Answer Set

Create instruction-answer pairs from the extracted chunks of text.

The `InstructionAnswerSet` class allows us to create instances directly from JSON strings, which is useful when parsing the output from the OpenAI API.

In [ ]:
class InstructionAnswerSet:
	def __init__(self, pairs: List[Tuple[str, str]]):
		self.pairs = pairs
		
	@classmethod
	def from_json(cls, json_str: str) -> 'InstructionAnswerSet':
		data = json.loads(json_str)
		pairs = [(pair['instruction'], pair['answer'])
			for pair in data['instruction_answer_pairs']]
		return cls(pairs)
		
	def __iter__(self):
		return iter(self.pairs)

# Use LLM to Transform into Instruction-Answer Pairs

In [ ]:
def generate_instruction_answer_pairs(
	extract: str, client: OpenAI
) -> List[Tuple[str, str]]:
	prompt = f"""Based on the following extract, generate five
instruction-answer pairs. Each instruction \
must ask to write about a specific topic contained in the context.
each answer \
must provide a relevant paragraph based on the information found in
the \
context. Only use concepts from the context to generate the
instructions. \
Instructions must never explicitly mention a context, a system, a
course, or an extract. \
Instructions must be self-contained and general. \
Answers must imitate the writing style of the context. \
Example instruction: Explain the concept of an LLM Twin. \
Example answer: An LLM Twin is essentially an AI character that
mimics your writing style, personality, and voice. \
It's designed to write just like you by incorporating these elements
into a language model. \
The idea is to create a digital replica of your writing habits using
advanced AI techniques. \
Provide your response in JSON format with the following structure:
{{
"instruction_answer_pairs": [
{{"instruction": "...", "answer": "..."}},
...
]
}}

Extract:
{extract}
"""

	completion = client.chat.completions.create(
		model=MODEL,
		messages=[
			{
				"role": "system", "content": "You are a helpful assistant who \
			generates instruction-answer pairs based on the given context. \
			Provide your response in JSON format.",
			},
			{"role": "user", "content": prompt},
		],
		response_format={"type": "json_object"},
		max_tokens=1200,
		temperature=0.7,
	)
	
	# Parse the structured output
	result = InstructionAnswerSet.from_json(completion.choices[0].message.content)
	
	# Convert to list of tuples
	return result.pairs

# Main Function

Call function to create our instruction dataset from the main function.

In [ ]:
def create_instruction_dataset(
	dataset: Dataset, client: OpenAI, num_workers: int = 4
) -> Dataset:
	extracts = extract_substrings(dataset)
	instruction_answer_pairs = []
	with concurrent.futures.ThreadPoolExecutor(max_workers=num_workers) as executor:
		futures = [executor.submit(generate_instruction_answer_pairs, extract, client)
			for extract in extracts
		]
		for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures)):
			instruction_answer_pairs.extend(future.result())
	
	instructions, answers = zip(*instruction_answer_pairs)
	return Dataset.from_dict(
		{"instruction": list(instructions), "output": list(answers)}
	)

In [ ]:
def main(dataset_id: str) -> Dataset:
	client = OpenAI()
	
	# 1. Load the raw data
	raw_dataset = load_articles_from_json(DATASET)
	print("Raw dataset:")
	print(raw_dataset.to_pandas())
	
	# 2. Create instructiondataset
	instruction_dataset = create_instruction_dataset(raw_dataset, client)
	print("Instruction dataset:")
	print(instruction_dataset.to_pandas())
	
	# 3. Train/test split and export
	filtered_dataset = instruction_dataset.train_test_split(test_size=0.1)
	filtered_dataset.push_to_hub("clanredhead/llmtwin")
	
	return filtered_dataset

In [ ]:
Dataset({
	features: ['instruction', 'output'],
	num_rows: 3335
})

In [ ]:
main("1")

In [ ]:
client = OpenAI()

In [ ]:
# 1. Load the raw data
raw_dataset = load_articles_from_json(DATASET)
print("Raw dataset:")
print(raw_dataset.to_pandas())

In [ ]:
# 2. Create instructiondataset
instruction_dataset = create_instruction_dataset(raw_dataset, client)
print("Instruction dataset:")
print(instruction_dataset.to_pandas())

In [ ]:
# 3. Train/test split and export
filtered_dataset = instruction_dataset.train_test_split(test_size=0.1)
filtered_dataset.push_to_hub("clanredhead/llmtwin")